# 🧠 Лабораторная работа 3. Loss, Gradient Descent и Backpropagation

Цель: увидеть один шаг обучения нейросети буквально по числам.

Мы посмотрим:
- MSE;
- BCEWithLogitsLoss;
- gradients;
- параметры до и после `optimizer.step()`;
- полный training loop;
- влияние `Learning Rate`.


# 1. Импорт библиотек

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

print("PyTorch:", torch.__version__)

# 2. MSE вручную

In [ ]:
prediction = torch.tensor(0.7)
target = torch.tensor(1.0)

mse_manual = (prediction - target) ** 2

print("Prediction:", prediction.item())
print("Target:", target.item())
print("MSE:", mse_manual.item())

# 3. MSE через PyTorch

In [ ]:
mse_loss = nn.MSELoss()

prediction_batch = torch.tensor([[0.7]])
target_batch = torch.tensor([[1.0]])

loss = mse_loss(prediction_batch, target_batch)
print("PyTorch MSE:", loss.item())

# 4. BCEWithLogitsLoss и logits

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

logit = torch.tensor([[1.5]])
target = torch.tensor([[1.0]])

loss = bce_loss(logit, target)
probability = torch.sigmoid(logit)

print("Logit:", logit.item())
print("Probability:", probability.item())
print("BCEWithLogitsLoss:", loss.item())

# 5. Учебный датасет «Светофор»

In [ ]:
features = torch.tensor(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0],
        [0.0, 1.0, 0.0],
        [0.0, 1.0, 1.0],
        [1.0, 0.0, 0.0],
        [1.0, 0.0, 1.0],
        [1.0, 1.0, 0.0],
        [1.0, 1.0, 1.0],
    ],
    dtype=torch.float32,
)

targets = torch.tensor(
    [[0.0], [1.0], [0.0], [1.0], [0.0], [0.0], [0.0], [0.0]],
    dtype=torch.float32,
)

print(features.shape, targets.shape)

# 6. Создаём маленькую модель

In [ ]:
class TrafficLightNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, x):
        return self.network(x)


torch.manual_seed(RANDOM_SEED)
model = TrafficLightNetwork()
print(model)

# 7. Параметры ДО обучения

In [ ]:
for name, parameter in model.named_parameters():
    print(name)
    print(parameter.data)
    print()

# 8. Один Forward Pass

In [ ]:
loss_function = nn.BCEWithLogitsLoss()

logits = model(features)
loss = loss_function(logits, targets)

print("Logits:")
print(logits)
print("\nLoss:", loss.item())

# 9. До backward() gradients отсутствуют

In [ ]:
for name, parameter in model.named_parameters():
    print(name, "grad =", parameter.grad)

# 10. Выполняем backward()

In [ ]:
model.zero_grad()
loss.backward()

for name, parameter in model.named_parameters():
    print("\n", name)
    print(parameter.grad)

# 11. Один конкретный вес

In [ ]:
first_layer = model.network[0]

weight_before = first_layer.weight.data[0, 0].item()
gradient_value = first_layer.weight.grad[0, 0].item()

print("Вес ДО optimizer.step():", weight_before)
print("Gradient этого веса:", gradient_value)

# 12. Один шаг SGD

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer.step()

weight_after = first_layer.weight.data[0, 0].item()

print("Вес ДО:", weight_before)
print("Gradient:", gradient_value)
print("Вес ПОСЛЕ:", weight_after)
print("Изменение:", weight_after - weight_before)

# 13. Проверяем формулу SGD

In [ ]:
expected_weight = weight_before - 0.1 * gradient_value

print("Ожидаемый вес:", expected_weight)
print("Фактический вес:", weight_after)
print("Совпадение:", abs(expected_weight - weight_after) < 1e-6)

# 14. Почему нужен zero_grad()

In [ ]:
new_logits = model(features)
new_loss = loss_function(new_logits, targets)

new_loss.backward()

print("Gradient после второго backward без zero_grad():")
print(first_layer.weight.grad[0, 0].item())

Gradients в PyTorch накапливаются, поэтому в обычном цикле перед `backward()` используется:

```python
optimizer.zero_grad()
```


# 15. Полный training loop

In [ ]:
def calculate_accuracy(model, features, targets):
    model.eval()

    with torch.no_grad():
        logits = model(features)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    return float((predictions == targets).float().mean().item())


def train_model(learning_rate=0.05, epochs=500):
    torch.manual_seed(RANDOM_SEED)

    model = TrafficLightNetwork()
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = []

    for _ in range(epochs):
        model.train()

        logits = model(features)
        loss = loss_function(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.item())

    accuracy = calculate_accuracy(model, features, targets)

    return model, history, accuracy

# 16. Обучаем модель

In [ ]:
trained_model, loss_history, accuracy = train_model()

print("Final Loss:", loss_history[-1])
print("Accuracy:", accuracy)

# 17. График Loss

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(loss_history)
plt.title("Loss во время обучения")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# 18. Сравниваем Learning Rate

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.5]

histories = {}
results = []

for lr in learning_rates:
    _, history, accuracy = train_model(learning_rate=lr, epochs=500)
    histories[lr] = history
    results.append({
        "Learning Rate": lr,
        "Final Loss": history[-1],
        "Accuracy": accuracy,
    })

pd.DataFrame(results).round(6)

# 19. Графики разных Learning Rate

In [ ]:
plt.figure(figsize=(11, 6))

for lr, history in histories.items():
    plt.plot(history, label=f"lr={lr}")

plt.title("Влияние Learning Rate на Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# 20. Gradient Descent вручную

In [ ]:
weight = 2.0
gradient = 0.4
learning_rate = 0.1

new_weight = weight - learning_rate * gradient

print("Старый вес:", weight)
print("Gradient:", gradient)
print("Learning Rate:", learning_rate)
print("Новый вес:", new_weight)

# 21. Gradient с отрицательным знаком

In [ ]:
weight = 2.0
gradient = -0.4
learning_rate = 0.1

new_weight = weight - learning_rate * gradient

print("Старый вес:", weight)
print("Gradient:", gradient)
print("Новый вес:", new_weight)

# 22. 📌 Что нужно запомнить

```text
Forward Pass
    ↓
Prediction
    ↓
Loss
    ↓
optimizer.zero_grad()
    ↓
loss.backward()
    ↓
parameter.grad
    ↓
optimizer.step()
    ↓
updated parameters
```

> `backward()` вычисляет gradients, а `step()` изменяет parameters.


# 23. ❓ Самопроверка

1. Чем Loss отличается от Accuracy?
2. Что такое Gradient?
3. Что хранится в `parameter.grad`?
4. Что делает `loss.backward()`?
5. Что делает `optimizer.step()`?
6. Почему нужен `optimizer.zero_grad()`?
7. Что такое Learning Rate?
8. Почему слишком большой Learning Rate может мешать обучению?
9. Почему Gradient Descent идёт против Gradient?
10. Что такое Backpropagation?


# 24. 🧩 Эксперименты

Попробуй:
- изменить `Learning Rate`;
- заменить Adam на SGD;
- уменьшить число Epoch;
- увеличить число скрытых нейронов;
- вывести `.grad` после разных Epoch;
- сравнить значение одного веса в начале и в конце обучения.
